# LearnMate AI: Meta Llama 3 8B Instruct Export Pipeline
## 4-bit Quantization (8da4w) + ExecuTorch v0.5.0 + XNNPACK Delegation

**Project:** LearnMate AI (Personalized On-Device AI Learning Companion)
**Target Hardware:** Qualcomm Snapdragon 662 / ARM64 Android (Redmi 9 Power 4GB RAM)
**Quantization:** TorchAO `8da4w` (INT4 groupwise weights, INT8 dynamic activations) + 4-bit embedding quantization
**Backend:** XNNPACK CPU Delegate (`-X` flag enabled for ARM NEON acceleration)
**Context Length:** 512 tokens (bounded for mobile KV cache conservation)
**Output:** `llama3_8b_8da4w.pte` (~4.27 GB) + `tokenizer.model`

> **Prerequisites on Kaggle:**
> 1. Set **Accelerator** to `None` or `GPU T4 x 2` (Kaggle provides ~30GB System RAM).
> 2. In Kaggle top menu: **Add-ons -> Secrets**, add secret with Label: `HF_TOKEN` and your Hugging Face read token.

In [ ]:
# Cell 1: Verify Kaggle Resources (RAM & Disk)
import psutil
import shutil
import os

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Total System RAM: {ram_gb:.2f} GB")

disk_working = shutil.disk_usage("/kaggle/working")
print(f"Disk Space in /kaggle/working: Total={disk_working.total / (1024**3):.2f} GB, Free={disk_working.free / (1024**3):.2f} GB")

temp_dir = "/kaggle/temp"
os.makedirs(temp_dir, exist_ok=True)
disk_temp = shutil.disk_usage(temp_dir)
print(f"Disk Space in /kaggle/temp: Total={disk_temp.total / (1024**3):.2f} GB, Free={disk_temp.free / (1024**3):.2f} GB")

assert ram_gb >= 24.0, f"Insufficient RAM ({ram_gb:.2f} GB). Export requires >= 24GB."

In [ ]:
# Cell 2: Install Pinned Dependencies (ExecuTorch v0.5.0 Stack)
!pip install -q --upgrade pip setuptools wheel
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q torchao==0.8.0 sentencepiece tiktoken blobfile huggingface_hub

import torch
import torchao
print(f"PyTorch: {torch.__version__}, TorchAO: {torchao.__version__}")
from torchao.quantization.quant_api import Int8DynActInt4WeightQuantizer
print("TorchAO Int8DynActInt4WeightQuantizer verified successfully.")

In [ ]:
# Cell 3: Clone ExecuTorch v0.5.0
!git clone --depth 1 --branch v0.5.0 https://github.com/pytorch/executorch.git /kaggle/temp/executorch
%cd /kaggle/temp/executorch
!pip install -q -e .

In [ ]:
# Cell 4: Authenticate & Download Llama 3 8B Checkpoint into Scratch Directory
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

checkpoint_dir = "/kaggle/temp/llama3_8b_raw"
os.makedirs(checkpoint_dir, exist_ok=True)

print("Downloading Meta Llama 3 8B Instruct consolidated checkpoint, params and tokenizer...")
snapshot_download(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    token=hf_token,
    local_dir=checkpoint_dir,
    allow_patterns=[
        "original/consolidated.00.pth",
        "original/params.json",
        "original/tokenizer.model"
    ]
)
print("Download completed successfully.")

In [ ]:
# Cell 5: Execute 4-bit Quantization (8da4w) and ExecuTorch Export with XNNPACK Delegate
import os
import sys
import time

checkpoint_dir = "/kaggle/temp/llama3_8b_raw"
checkpoint_path = os.path.join(checkpoint_dir, "original", "consolidated.00.pth")
params_path = os.path.join(checkpoint_dir, "original", "params.json")

output_dir = "/kaggle/working"
pte_output = os.path.join(output_dir, "llama3_8b_8da4w.pte")

cmd = (
    f"{sys.executable} -m examples.models.llama.export_llama "
    f"--model llama3 "
    f"--checkpoint '{checkpoint_path}' "
    f"--params '{params_path}' "
    f"-kv "
    f"--use_sdpa_with_kv_cache "
    f"-qmode 8da4w "
    f"--group_size 128 "
    f"--embedding-quantize 4,32 "
    f"--max_seq_length 512 "
    f"-X "
    f"--metadata '{{\"get_bos_id\": 128000, \"get_eos_ids\": [128001, 128009]}}' "
    f"--output_name '{pte_output}'"
)

print(f"Executing export command:\n{cmd}\n")
start_time = time.time()
!{cmd}
elapsed = time.time() - start_time
print(f"\nExport completed in {elapsed / 60:.2f} minutes.")

In [ ]:
# Cell 6: Copy Tokenizer and Verify Generated Artifacts
import os
import shutil
import hashlib

pte_file = "/kaggle/working/llama3_8b_8da4w.pte"
assert os.path.exists(pte_file), f"Export failed! {pte_file} does not exist."

size_bytes = os.path.getsize(pte_file)
size_gb = size_bytes / (1024**3)
print(f"=== EXPORT SUCCESSFUL ===")
print(f"Artifact: {pte_file}")
print(f"Size: {size_gb:.2f} GB ({size_bytes:,} bytes)")

# Copy tokenizer files to /kaggle/working
tok_model_src = "/kaggle/temp/llama3_8b_raw/original/tokenizer.model"
tok_json_src = "/kaggle/temp/llama3_8b_raw/tokenizer.json"

if os.path.exists(tok_model_src):
    shutil.copy(tok_model_src, "/kaggle/working/tokenizer.model")
    print("Copied tokenizer.model to /kaggle/working/")
if os.path.exists(tok_json_src):
    shutil.copy(tok_json_src, "/kaggle/working/tokenizer.json")
    print("Copied tokenizer.json to /kaggle/working/")

# Calculate MD5 Checksum
print("Calculating MD5 checksum...")
hasher = hashlib.md5()
with open(pte_file, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
        hasher.update(chunk)
print(f"MD5: {hasher.hexdigest()}")

print("\nDone! You can now download llama3_8b_8da4w.pte and tokenizer.model from the Kaggle Output panel.")